# Populate Bigtable — Continuous Inserts, Updates, and Deletes

This notebook writes a **continuous stream** of mutations to a Bigtable table:
- **Insert** — `set_cell` (new or overwrite)
- **Update** — `set_cell` again on the same row
- **Delete** — delete cells in a column, or delete the entire row

Job parameters use widgets; all other settings are local variables in the next cell.

## Install package (if needed)

On **Databricks**: run the cell below, then re-attach the cluster if needed.

In [ ]:
# When running via bundle/job, libraries are pre-installed via cluster config.
# For interactive use, uncomment and run:
# %pip install -q google-cloud-bigtable
# %restart_python

import google.cloud.bigtable
print(f"google-cloud-bigtable: OK")

## Configuration

Run this cell to set parameters. **Job parameters** (project_id, instance_id, table_id) come from Databricks widgets—when the notebook runs as a job, the job passes these via base_parameters. **All other variables** are local (capitalised); edit them in this cell. Then run **Create table (if it doesn't exist)** and **Run the continuous populate loop** in order.

In [ ]:
dbutils.widgets.text("project_id", "", "GCP Project ID")
dbutils.widgets.text("instance_id", "", "Bigtable Instance ID")
dbutils.widgets.text("table_id", "", "Bigtable Table ID")
dbutils.widgets.text("credentials_file", "", "Google Application Credentials File")
dbutils.widgets.text("run_duration_seconds", "0", "Run duration (0 = infinite)")

project_id = dbutils.widgets.get("project_id")
instance_id = dbutils.widgets.get("instance_id")
table_id = dbutils.widgets.get("table_id")
credentials_file = dbutils.widgets.get("credentials_file")

import os

if credentials_file == "":
    import glob
    gcp_jsons = glob.glob(os.path.join(os.getcwd(), "gcp*.json"))
    if gcp_jsons:
        credentials_file = gcp_jsons[0]

if credentials_file:
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = credentials_file

# Local variables (not passed by job)
BIGTABLE_COLUMN_FAMILY = "cf1"
BIGTABLE_COLUMN_FAMILY_2 = "cf2"
RUN_DURATION_SECONDS = int(dbutils.widgets.get("run_duration_seconds"))
BATCH_SIZE = 100
BATCH_DELAY_SECONDS = 1.0
ROW_KEY_PREFIX = "row"
NUM_ROWS = 500
INSERT_PCT = 50
UPDATE_PCT = 30
DELETE_PCT = 20
COLUMN_QUALIFIER = "payload"
COLUMN_QUALIFIER_2 = "data"
VALUE_SIZE_BYTES = 100
DELETE_TYPE = "column"

# Show job params for verification
{"project_id": project_id, "instance_id": instance_id, "table_id": table_id, "credentials_file": credentials_file, "run_duration_seconds": RUN_DURATION_SECONDS}

if not project_id or not instance_id or not table_id:
    raise ValueError("Set project_id, instance_id, and table_id in the widget UI.")

## Create table (if it doesn't exist)

Run the cell below to create the Bigtable table with **change stream** enabled and **7-day retention**. The instance must already exist.

In [ ]:
# Create table with change stream (7-day retention) if it doesn't exist
from google.cloud.bigtable import Client
from google.cloud.bigtable_admin_v2.types import table as gba_table
from google.cloud.bigtable_admin_v2.types import bigtable_table_admin
from google.protobuf import duration_pb2

column_family = BIGTABLE_COLUMN_FAMILY
column_family_2 = BIGTABLE_COLUMN_FAMILY_2

client = Client(project=project_id, admin=True)
instance = client.instance(instance_id)
table_obj = instance.table(table_id)

if table_obj.exists():
    print(f"Table '{table_id}' already exists.")
else:
    print(f"Creating table '{table_id}' with column families '{column_family}', '{column_family_2}' and change stream (7-day retention)...")
    table_admin = client.table_admin_client
    parent = instance.name
    retention_seconds = 7 * 24 * 3600
    change_stream_config = gba_table.ChangeStreamConfig(
        retention_period=duration_pb2.Duration(seconds=retention_seconds),
    )
    table_pb = gba_table.Table(
        column_families={
            column_family: gba_table.ColumnFamily(),
            column_family_2: gba_table.ColumnFamily(),
        },
        change_stream_config=change_stream_config,
    )
    request = bigtable_table_admin.CreateTableRequest(
        parent=parent,
        table_id=table_id,
        table=table_pb,
    )
    table_admin.create_table(request=request)
    print("Table created with change stream enabled.")

## Run the continuous populate loop

Runs the insert/update/delete loop using the configured parameters. Stops on **KeyboardInterrupt** (Ctrl+C).

In [ ]:
# Run the continuous populate loop
import base64
import json
import os
import random
import time
from google.cloud.bigtable import Client

column_family = BIGTABLE_COLUMN_FAMILY
column_family_2 = BIGTABLE_COLUMN_FAMILY_2
column_families = [column_family, column_family_2]
def _enc(s):
    return s.encode("utf-8") if isinstance(s, str) else s
column_qualifier_by_cf = {
    column_family: _enc(COLUMN_QUALIFIER),
    column_family_2: _enc(COLUMN_QUALIFIER_2),
}
run_duration_seconds = RUN_DURATION_SECONDS
batch_size = BATCH_SIZE
batch_delay_seconds = BATCH_DELAY_SECONDS
row_key_prefix = ROW_KEY_PREFIX
num_rows = NUM_ROWS
insert_pct = INSERT_PCT
update_pct = UPDATE_PCT
delete_pct = DELETE_PCT
value_size_bytes = VALUE_SIZE_BYTES
delete_type = (DELETE_TYPE or "column").strip().lower()

if insert_pct + update_pct + delete_pct != 100:
    raise ValueError("INSERT_PCT + UPDATE_PCT + DELETE_PCT must sum to 100")

client = Client(project=project_id, admin=True)
table = client.instance(instance_id).table(table_id)
row_keys = [f"{row_key_prefix}-{i}".encode("utf-8") for i in range(num_rows)]
deadline = (time.time() + run_duration_seconds) if run_duration_seconds > 0 else None
op_counts = {"insert": 0, "update": 0, "delete": 0}
start = time.time()
batch_num = 0

while True:
    if deadline and time.time() >= deadline:
        break
    batch_counts = {"insert": 0, "update": 0, "delete": 0}
    for _ in range(batch_size):
        r = random.randint(1, 100)
        cf = random.choice(column_families)
        cq = column_qualifier_by_cf[cf]
        if r <= insert_pct:
            row_key = random.choice(row_keys)
            written_at = time.time()
            payload = base64.b64encode(os.urandom(max(0, value_size_bytes - 60))).decode("ascii")
            value = json.dumps({"written_at": written_at, "p": payload}).encode("utf-8")
            row = table.direct_row(row_key)
            row.set_cell(cf, cq, value)
            row.commit()
            op_counts["insert"] += 1
            batch_counts["insert"] += 1
        elif r <= insert_pct + update_pct:
            row_key = random.choice(row_keys)
            written_at = time.time()
            payload = base64.b64encode(os.urandom(max(0, value_size_bytes - 60))).decode("ascii")
            value = json.dumps({"written_at": written_at, "p": payload}).encode("utf-8")
            row = table.direct_row(row_key)
            row.set_cell(cf, cq, value)
            row.commit()
            op_counts["update"] += 1
            batch_counts["update"] += 1
        else:
            row_key = random.choice(row_keys)
            row = table.direct_row(row_key)
            if delete_type == "row":
                row.delete()
                row.commit()
            else:
                row.delete_cells(cf, [cq])
                row.commit()
            op_counts["delete"] += 1
            batch_counts["delete"] += 1
    batch_num += 1
    print(f"Batch {batch_num}: insert={batch_counts['insert']} update={batch_counts['update']} delete={batch_counts['delete']} (total {batch_size})")
    if batch_delay_seconds > 0:
        time.sleep(batch_delay_seconds)

client.close()

elapsed = time.time() - start
total = sum(op_counts.values())
print(f"Run finished: {elapsed:.1f}s, batches={batch_num}, ops: insert={op_counts['insert']} update={op_counts['update']} delete={op_counts['delete']} total={total}")